# Set up environment

In [1]:
import sys
from pathlib import Path


def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False


def clone_repository() -> None:
    !git clone https://github.com/decodingml/hands-on-recommender-system.git
    %cd hands-on-recommender-system/


def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()

    root_dir = str(Path().absolute())
    print("⛳️ Google Colab environment")
else:
    root_dir = str(Path().absolute().parent)
    print("⛳️ Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    print(f"Adding the following directory to the PYTHONPATH: {root_dir}")
    sys.path.append(root_dir)

⛳️ Local environment
Adding the following directory to the PYTHONPATH: c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course


# Scheduling Hopsworks materialization jobs 


## Imports

In [2]:
from datetime import datetime, timezone

from recsys import hopsworks_integration

from recsys.config import settings

c:\Users\PC\OneDrive - National Economics University\Tài liệu\personalized-recommender-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Connect to Hopsworks Feature Store 

In [3]:
import hopsworks

hopsworks_host = "eu-west.cloud.hopsworks.ai"

project = hopsworks.login(
    host=hopsworks_host,
    api_key_value=settings.HOPSWORKS_API_KEY.get_secret_value()
)

# Thêm logic này để tự tạo project nếu tài khoản bị trống
if project is None:
    project = hopsworks.create_project("recommender_system", description="H&M Recommender")

fs = project.get_feature_store()

jobs_api = project.get_jobs_api()

2026-03-11 02:17:53,341 INFO: Initializing external client
2026-03-11 02:17:53,343 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-03-11 02:17:56,214 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31873


# Retrieving materialization jobs


In [4]:
interactions_job = jobs_api.get_job("interactions_1_offline_fg_materialization")
interactions_job

Job('interactions_1_offline_fg_materialization', 'SPARK')

In [5]:
transactions_job = jobs_api.get_job("transactions_1_offline_fg_materialization")
transactions_job

Job('transactions_1_offline_fg_materialization', 'SPARK')

# Running materialization jobs


In [6]:
interactions_job_execution = interactions_job.run()
interactions_job_execution

Launching job: interactions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31873/jobs/named/interactions_1_offline_fg_materialization/executions
2026-03-11 02:18:08,081 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-03-11 02:18:11,391 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-03-11 02:20:03,746 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-03-11 02:20:04,509 INFO: Waiting for log aggregation to finish.
2026-03-11 02:20:04,515 INFO: Execution finished successfully.


Execution('SUCCEEDED', 'FINISHED', '2026-03-10T19:17:59.000Z', '-op offline_fg_materialization -path hdfs:///Projects/recommender_system/Resources/jobs/interactions_1_offline_fg_materialization/config_1773119583763')

In [7]:
transactions_job_execution = transactions_job.run()
transactions_job_execution

Launching job: transactions_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31873/jobs/named/transactions_1_offline_fg_materialization/executions
2026-03-11 02:20:14,067 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2026-03-11 02:20:17,362 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2026-03-11 02:22:09,664 INFO: Waiting for execution to finish. Current state: FINISHED. Final status: SUCCEEDED
2026-03-11 02:22:10,438 INFO: Waiting for log aggregation to finish.
2026-03-11 02:22:10,439 INFO: Execution finished successfully.


Execution('SUCCEEDED', 'FINISHED', '2026-03-10T19:20:05.000Z', '-op offline_fg_materialization -path hdfs:///Projects/recommender_system/Resources/jobs/transactions_1_offline_fg_materialization/config_1773119393642')

## Scheduling materialization jobs


In [8]:
interactions_job.schedule(
    cron_expression="0 0 0 * * ?",  # Runs at midnight (00:00:00) every day
    start_time=datetime.now(tz=timezone.utc),
)
interactions_job.job_schedule.next_execution_date_time

datetime.datetime(2026, 3, 11, 0, 0, tzinfo=datetime.timezone.utc)

In [9]:
transactions_job.schedule(
    cron_expression="0 0 0 * * ?",  # Runs at midnight (00:00:00) every day
    start_time=datetime.now(tz=timezone.utc),
)
transactions_job.job_schedule.next_execution_date_time

datetime.datetime(2026, 3, 11, 0, 0, tzinfo=datetime.timezone.utc)